# 9주차 ① 전이학습 기준선 — 실습 1~3

**목표**: 본인 이미지를 `ImageFolder` 로 로딩하고, 사전학습 ViT 의 헤드를 내 클래스 수로 교체한 뒤
**백본을 얼려 학습**시켜 오늘 모든 비교의 **기준선**을 만든다.

> **준비물**: `mydata/train|val/<클래스>/` 구조의 본인 이미지.
> 먼저 터미널에서 `python prepare_mydata.py` 로 폴더 구조를 검증하세요.

> ⚠️ **GPU 가 없다면 지금 Colab 으로 옮기세요.** 이번 주차는 학기에서 가장 무겁고,
> CPU 로는 실습 3·5가 **사실상 불가능**합니다. 늦게 옮기면 3교시 비교표를 못 만듭니다.
> ```python
> from google.colab import drive
> drive.mount('/content/drive')
> DATA = "/content/drive/MyDrive/mydata"
> ```

```
   ViT-base 파라미터 8,600만 개   vs   내 데이터 200장
        → 절대 학습되지 않는다. 외우기만 하고 끝난다 (6주차 과적합의 극단 버전)

   [사전학습 (Pre-training)]                [미세조정 (Fine-tuning)]
   ImageNet 수천만 장으로                    내 데이터 200장으로
   "보는 눈"을 만든다                        "내 문제"에만 맞춘다
        │                                          ↑
        └───────  가중치를 그대로 가져온다 ────────┘
```

> **핵심 메시지 ★**: **전이학습이 적은 데이터에서 통하는 이유**는,
> 저수준 특징(선·모서리·질감)이 **문제가 달라도 거의 같기** 때문입니다.
> 7주차 실습 3에서 특징맵을 봤죠 — **1·2층이 배우는 것은 어느 데이터에서나 비슷합니다.**

| 방식 | 학습 대상 | 데이터가 | 자원이 | 성능 |
|---|---|---|---|---|
| **백본 동결** | 헤드만 (수천 개) | 적어도 됨 | 아주 적게 | 보통 |
| **LoRA** ★ | 어댑터만 (0.1~1%) | 적어도 됨 | 적게 | **전체에 근접** |
| **전체 미세조정** | 전부 (8,600만) | 많이 필요 | 많이 | 최고 |

## 실습 1 — 내 데이터셋 로딩

In [ ]:
# 셀 1 — 폴더 구조 검증
import torch, os
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision.transforms import v2
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

DATA = "mydata"          # Colab 이면 Drive 경로로
device = "cuda" if torch.cuda.is_available() else "cpu"
print("장치 :", device)

for split in ["train", "val"]:
    p = os.path.join(DATA, split)
    print(f"\n[{split}]")
    for c in sorted(os.listdir(p)):
        print(f"  {c:12s} {len(os.listdir(os.path.join(p, c))):3d}장")

> **막히면**:
> | 증상 | 조치 |
> |---|---|
> | `FileNotFoundError` | 경로 확인. Colab 이면 Drive 마운트 |
> | 클래스가 1개로 잡힌다 | `train/` 바로 아래에 이미지를 넣었다. **클래스 폴더**가 필요 |
> | `.HEIC` 를 못 읽는다 | `.jpg` 로 변환. 윈도우 사진 앱에서 일괄 가능 |
> | 데이터를 못 모아 왔다 | **대체 데이터셋 배포본**을 받으세요 (감점 없음) |
>
> 위 문제를 한 번에 짚어 주는 스크립트가 `prepare_mydata.py` 입니다.

In [ ]:
# 셀 2 — 전처리와 DataLoader
IMG = 224                                   # ★ ViT-base 가 기대하는 크기
MEAN, STD = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)     # ViT 계열의 표준값

train_tf = v2.Compose([
    v2.RandomResizedCrop(IMG, scale=(0.8, 1.0)),
    v2.RandomHorizontalFlip(),
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])
val_tf = v2.Compose([                        # ★ 증강 없음 (7주차)
    v2.Resize((IMG, IMG)),
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])

train_set = ImageFolder(f"{DATA}/train", transform=train_tf)
val_set   = ImageFolder(f"{DATA}/val",   transform=val_tf)
CLASSES = train_set.classes
NUM_CLASSES = len(CLASSES)

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)   # ★ 8GB 기준 16
val_loader   = DataLoader(val_set,   batch_size=16)

print("클래스 :", CLASSES)
xb, yb = next(iter(train_loader))
print("배치 shape :", xb.shape)             # (16, 3, 224, 224)

> **관찰 포인트 ★**: 7주차 CIFAR-10 은 `(B,3,32,32)` 였습니다. 오늘은 **`(B,3,224,224)`** 입니다.
> 픽셀 수가 **49배**입니다. 그래서 batch 를 128 → **16** 으로 줄였습니다.
> **3주차의 "GPU 는 빠르지만 좁다"** 가 여기서 실제 제약이 됩니다.

In [ ]:
# 셀 3 — 이미지 확인
fig, ax = plt.subplots(1, 6, figsize=(14, 2.6))
for i in range(6):
    img = xb[i].permute(1,2,0) * torch.tensor(STD) + torch.tensor(MEAN)
    ax[i].imshow(img.clamp(0,1)); ax[i].set_title(CLASSES[yb[i]]); ax[i].axis("off")
plt.tight_layout(); plt.show()

## 실습 2 — 모델 로드 + 헤드 교체

```
   [사전학습 ViT]                          [내 모델]
   입력 → [ 백본 (특징 추출) ] → 헤드(1000 클래스)     ImageNet 용
                  │                        ✗ 버린다
                  └──────────────→ 헤드(3~5 클래스)   ★ 새로 만들어 학습
```

In [ ]:
# 셀 4 — 두 가지 로딩 방식
import timm
from transformers import ViTForImageClassification

# ① timm — 한 줄
resnet = timm.create_model("resnet18", pretrained=True, num_classes=NUM_CLASSES)
print("timm ResNet18 파라미터 :", sum(p.numel() for p in resnet.parameters()))

# ② HuggingFace — 헤드 크기가 다르므로 명시가 필요하다
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,        # ★ 1000 → NUM_CLASSES 로 갈아 끼운다
).to(device)
print("HF ViT-base 파라미터 :", sum(p.numel() for p in model.parameters()))

> **핵심 메시지 ★ (출제 지점)**: **`ignore_mismatched_sizes=True` 가 필요한 이유** —
> 저장된 가중치의 분류 헤드는 **1000 클래스용**입니다. 우리는 3~5 클래스가 필요하죠.
> shape 이 다르니 그대로는 못 넣습니다. 이 옵션이 *"헤드만 버리고 새로 초기화하라"* 는 뜻입니다.
> **경고 메시지가 뜨는 것이 정상**입니다.

| | timm | HuggingFace |
|---|---|---|
| 로딩 | `create_model(..., num_classes=N)` 한 줄 | `from_pretrained(..., num_labels=N, ignore_mismatched_sizes=True)` |
| 강점 | 이미지 모델이 압도적으로 많다 | **텍스트·멀티모달까지 같은 방식** (12·14주차) |
| 출력 | 텐서 | **객체** (`.logits` 로 꺼낸다) ★ |

> **함정 ★**: HuggingFace 모델은 `model(xb)` 가 **텐서가 아니라 객체**를 반환합니다.
> **`model(xb).logits`** 로 꺼내야 합니다. 오늘 내내 나오는 실수입니다.

> ⚠️ **모델이 330MB 입니다.** 사전 캐시가 없으면 몇 분 걸립니다.

In [ ]:
# 셀 5 (참고) — 출력이 텐서가 아니라 객체라는 것 확인
with torch.no_grad():
    out = model(xb[:2].to(device))
print("반환 타입 :", type(out).__name__)
print("꺼낼 수 있는 것 :", [k for k in out.keys()])
print("out.logits.shape :", out.logits.shape)      # (2, NUM_CLASSES)

## 실습 3 — 백본 동결 학습 = 기준선

In [ ]:
# 셀 6 — 백본 동결
for p in model.vit.parameters():          # ★ 백본 전체
    p.requires_grad = False               #   기울기를 추적하지 않는다

for p in model.classifier.parameters():   # 헤드만
    p.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"학습 파라미터 {trainable:,} / 전체 {total:,}  ({trainable/total*100:.4f}%)")

> **핵심 메시지 ★**: `requires_grad = False` — **4주차에 배운 그 플래그**입니다.
> 끄면 `.grad` 가 안 채워지고, 옵티마이저가 갱신하지 않습니다.
> *"얼린다"* 는 말의 실체가 이 한 줄입니다.

> **핵심 메시지 ★★**: 7주차에 CNN 을 직접 짜 봤기 때문에 **이 말이 무슨 뜻인지 압니다.**
> 얼린다는 건 **여러분이 만들었던 그 Conv 층들의 가중치를 갱신하지 않는다**는 뜻입니다.

> **관찰 포인트**: 학습 파라미터가 전체의 **0.01% 미만**으로 나옵니다.
> 이 숫자를 **적어 두세요** — 3교시 비교표의 첫 줄입니다.

In [ ]:
# 셀 7 — 기준선 학습
import torch.nn as nn, time
from torch.amp import autocast, GradScaler

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=1e-3)   # ★ 켜진 것만 넘긴다
scaler = GradScaler(device, enabled=(device == "cuda"))            # 7주차 AMP

EPOCHS = 5        # ★ 실습실 실측에 맞춰 조정

def evaluate(m):
    m.eval(); c = t = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            with autocast(device, enabled=(device == "cuda")):
                out = m(xb).logits                    # ★ .logits
            c += (out.argmax(1) == yb).sum().item(); t += yb.size(0)
    return c / t

if device == "cuda": torch.cuda.reset_peak_memory_stats()
t0 = time.time()

for epoch in range(EPOCHS):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        with autocast(device, enabled=(device == "cuda")):
            loss = loss_fn(model(xb).logits, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
    print(f"epoch {epoch+1}/{EPOCHS} | 검증 정확도 {evaluate(model)*100:5.2f}%")

baseline = dict(
    방식="백본 동결",
    학습파라미터=trainable,
    정확도=evaluate(model),
    시간=time.time()-t0,
    VRAM=torch.cuda.max_memory_allocated()/1024**3 if device=="cuda" else 0,
)
print("\n기준선 :", baseline)

In [ ]:
# 셀 8 — 기준선을 파일로 남긴다  ★ 3교시 비교표에서 다시 읽는다
import json
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

torch.save(model.classifier.state_dict(), "models/vit_frozen_head.pt")

with open("results/baseline.json", "w", encoding="utf-8") as f:
    json.dump({**baseline, "클래스": CLASSES}, f, ensure_ascii=False, indent=2)

print("저장 완료")
print("  models/vit_frozen_head.pt")
print("  results/baseline.json     ← 3교시 실습 7 에서 읽는다")

> **관찰 포인트 ★**: 헤드만 학습하는데도 **정확도가 꽤 높게** 나옵니다.
> 백본이 이미 좋은 특징을 뽑아 주기 때문입니다.

> **핵심 메시지**: 이 결과가 **비교의 기준선**입니다. 2교시 LoRA, 3교시 4bit 가
> 이것보다 나은지 나쁜지를 **이 숫자에 대고** 판단합니다.
> 커널을 재시작해도 `results/baseline.json` 에 남아 있으니 안심하세요.

> **막히면**:
> | 증상 | 원인 |
> |---|---|
> | `'ImageClassifierOutput' object has no attribute 'argmax'` | `.logits` 를 빼먹었다 ★ |
> | `CUDA out of memory` | batch 를 16 → 8 로 |
> | 정확도가 안 오른다 | 옵티마이저에 **켜진 파라미터만** 넘겼는지 확인 |
> | 아주 느리다 | CPU 다. **지금 Colab 으로 옮기세요** |

---

### 이 노트북 체크리스트

- [ ] 전이학습이 적은 데이터에서 통하는 이유를 말할 수 있다 ★
- [ ] 내 데이터를 `ImageFolder` 로 로딩했다
- [ ] 224×224 · batch 16 인 이유를 안다
- [ ] `ignore_mismatched_sizes=True` 가 필요한 이유를 안다 ★
- [ ] HuggingFace 출력에서 `.logits` 를 꺼내야 하는 것을 안다 ★
- [ ] `requires_grad=False` 로 백본을 얼렸다
- [ ] 학습 파라미터 비율(0.01% 미만)을 확인했다
- [ ] **기준선 정확도·VRAM·시간을 기록**했다 ★